# 01 · Zarr Reorganisation
Reorganises the multiplexed fixed-cycle zarr stores into an OME-NGFF compliant structure,
then reads Harmony metadata and assay layout XML to populate `.zattrs` with acquisition
and experimental metadata per field-of-view.

Steps:
1. Move `*_plexed.zarr` stores into a unified `multiplexed.zarr` with per-FOV subdirectories
2. Read Harmony metadata and assay layout XML
3. Write OME-NGFF compliant `.zattrs` per FOV with channel, spatial, and experimental metadata

In [ ]:
import zarr
import shutil
import os
import json
import glob
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
from macrohet import dataio


def safe(x):
    """Sanitise a value for JSON serialisation - converts NaN/NA to None."""
    return None if pd.isna(x) else x


def load_assay_layout(xml_file_name):
    """
    Parse a Harmony assay layout XML file into a DataFrame indexed by well position.

    Args:
        xml_file_name (str): Path to the assay layout XML file.

    Returns:
        pd.DataFrame: Rows indexed by position string e.g. "(3, 4)", columns are layer names.
    """
    df_assay_layout = pd.DataFrame()
    if not os.path.exists(xml_file_name):
        print(f"Error: '{xml_file_name}' not found.")
        return df_assay_layout

    try:
        tree = ET.parse(xml_file_name)
        root = tree.getroot()
        namespaces = {'ns': 'http://www.perkinelmer.com/PEHH/HarmonyV5'}
        plate_data = {}

        for layer in root.findall('ns:Layer', namespaces):
            layer_name_el = layer.find('ns:Name', namespaces)
            if layer_name_el is None or layer_name_el.text is None:
                print("Warning: Layer without a name, skipping.")
                continue
            layer_name = layer_name_el.text.strip()

            for well in layer.findall('ns:Well', namespaces):
                row_el = well.find('ns:Row', namespaces)
                col_el = well.find('ns:Col', namespaces)
                val_el = well.find('ns:Value', namespaces)
                if row_el is None or col_el is None:
                    continue
                try:
                    row = int(row_el.text.strip())
                    col = int(col_el.text.strip())
                except ValueError:
                    continue
                value = val_el.text.strip() if val_el is not None and val_el.text else pd.NA
                plate_data.setdefault((row, col), {})[layer_name] = value

        if plate_data:
            df_assay_layout = pd.DataFrame.from_dict(plate_data, orient='index')
            df_assay_layout.index.names = ['Row', 'Col']
            df_assay_layout = df_assay_layout.sort_index().dropna(how='all')
            df_assay_layout['position'] = df_assay_layout.index.map(lambda rc: f"({rc[0]}, {rc[1]})")
            df_assay_layout = df_assay_layout.set_index('position').drop(columns=['Row', 'Col'])
        else:
            print("No data extracted from XML.")

    except ET.ParseError as e:
        print(f"XML parse error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

    return df_assay_layout

## 1 · Reorganise multiplexed zarr into NGFF structure
Moves each `*_plexed.zarr` into a `<fov>/images/` subdirectory within `multiplexed.zarr`,
and writes a minimal `.zattrs` per FOV root.

In [ ]:
source_root      = '/mnt/DATA3/BPP0050/multiplexed'           # flat *_plexed.zarr stores
destination_root = '/mnt/DATA3/BPP0050/multiplexed.zarr'     # NGFF-compliant output

os.makedirs(destination_root, exist_ok=True)

for name in tqdm(os.listdir(source_root)):
    if not name.endswith('_plexed.zarr'):
        continue

    fov_name = name.replace('_plexed.zarr', '')   # e.g. '2_4'
    src      = os.path.join(source_root, name)
    dst      = os.path.join(destination_root, fov_name)
    dst_img  = os.path.join(dst, 'images')
    os.makedirs(dst_img, exist_ok=True)

    # Move all zarr chunk files and metadata into images/
    for fname in os.listdir(src):
        src_f = os.path.join(src, fname)
        dst_f = os.path.join(dst_img, fname)
        if os.path.exists(dst_f):
            print(f"Skipping existing: {dst_f}")
            continue
        shutil.move(src_f, dst_f)

    # Write FOV-level .zattrs
    zattrs_fov = {
        "multiscales": [{
            "version": "0.4",
            "datasets": [{"path": "images"}],
            "axes": ["round", "time", "channel", "z", "y", "x"],
            "name": fov_name
        }],
        "axes": [
            {"name": "round",   "type": "other"},
            {"name": "time",    "type": "time"},
            {"name": "channel", "type": "channel"},
            {"name": "z",       "type": "space"},
            {"name": "y",       "type": "space"},
            {"name": "x",       "type": "space"}
        ]
    }
    with open(os.path.join(dst, '.zattrs'), 'w') as f:
        json.dump(zattrs_fov, f, indent=2)

print("Reorganisation complete.")

## 2 · Load Harmony metadata and assay layout
Using Cy6 as the representative acquisition for channel and spatial metadata extraction.

In [ ]:
base_dir = '/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Fixed_Cy6__2025-04-19T16_14_26-Measurement 1'

metadata_fn = os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
metadata    = dataio.read_harmony_metadata(metadata_fn)

assay_layout_fn = glob.glob(os.path.join(base_dir, 'acquisition/Assaylayout/*.xml'))[0]
assay_layout    = load_assay_layout(assay_layout_fn)

print(f"Metadata columns: {list(metadata.keys())}")
print(f"Assay layout positions: {list(assay_layout.index)}")

## 3 · Write OME-NGFF compliant `.zattrs` per FOV
Populates each FOV's `.zattrs` with:
- Multiscales axes with physical pixel spacing (um)
- Channel metadata (name, wavelengths, exposure)
- Experimental conditions from the assay layout (infection status, antibiotic treatment)

In [ ]:
# Extract physical pixel spacings from metadata
z_um         = metadata['PositionZ'].astype(float) * 1e6
z_unique     = np.sort(z_um.dropna().unique())
z_spacing_um = float(np.mean(np.diff(z_unique)))
y_spacing_um = float(metadata['ImageResolutionY'].astype(float).iloc[0] * 1e6)
x_spacing_um = float(metadata['ImageResolutionX'].astype(float).iloc[0] * 1e6)

# Group metadata by FOV and channel
channel_grouped = (
    metadata.groupby(['Row', 'Col', 'ChannelID'])
    .agg({
        'ChannelName':              'first',
        'ExposureTime':             'first',
        'ImageResolutionX':         'first',
        'ImageResolutionY':         'first',
        'ImageType':                'first',
        'ChannelType':              'first',
        'IlluminationType':         'first',
        'AcquisitionType':          'first',
        'CameraType':               'first',
        'BinningX':                 'first',
        'BinningY':                 'first',
        'MainExcitationWavelength': 'first',
        'MainEmissionWavelength':   'first',
        'ObjectiveMagnification':   'first',
        'ObjectiveNA':              'first'
    })
    .reset_index()
)

metadata_dir = '/mnt/DATA3/BPP0050/multiplexed.zarr'

for (row, col), group in channel_grouped.groupby(['Row', 'Col']):
    fov_path    = os.path.join(metadata_dir, f'{row}_{col}')
    zattrs_path = os.path.join(fov_path, '.zattrs')
    os.makedirs(fov_path, exist_ok=True)

    # Per-channel metadata
    channels = [
        {
            'id':                    int(ch_row['ChannelID']),
            'name':                  ch_row['ChannelName'],
            'exposure_time':         ch_row['ExposureTime'],
            'excitation_wavelength': ch_row['MainExcitationWavelength'],
            'emission_wavelength':   ch_row['MainEmissionWavelength'],
            'channel_type':          ch_row['ChannelType']
        }
        for _, ch_row in group.iterrows()
    ]

    # Experimental conditions from assay layout
    try:
        condition            = assay_layout.loc[str((int(row), int(col)))]
        infection_status     = safe(condition.get('Infection status', None))
        antibiotic_treatment = safe(condition.get('Antibiotic treatment', None))
    except KeyError:
        infection_status     = None
        antibiotic_treatment = None

    zattrs = {
        'multiscales': [{
            'version': '0.4',
            'datasets': [{
                'path': 'images',
                'coordinateTransformations': [{
                    'type': 'scale',
                    'scale': [1, 1, 1, z_spacing_um, y_spacing_um, x_spacing_um]
                }]
            }],
            'axes': [
                {'name': 'round',   'type': 'other'},
                {'name': 'time',    'type': 'time'},
                {'name': 'channel', 'type': 'channel'},
                {'name': 'z',       'type': 'space', 'unit': 'micrometer'},
                {'name': 'y',       'type': 'space', 'unit': 'micrometer'},
                {'name': 'x',       'type': 'space', 'unit': 'micrometer'}
            ]
        }],
        'acquisition_metadata': {
            'infection_status':        infection_status,
            'antibiotic_treatment':    antibiotic_treatment,
            'objective_magnification': group['ObjectiveMagnification'].iloc[0],
            'objective_na':            group['ObjectiveNA'].iloc[0],
            'camera':                  group['CameraType'].iloc[0],
            'binning':                 {'x': group['BinningX'].iloc[0], 'y': group['BinningY'].iloc[0]},
            'image_type':              group['ImageType'].iloc[0],
            'illumination_type':       group['IlluminationType'].iloc[0],
            'acquisition_type':        group['AcquisitionType'].iloc[0],
            'channels':                channels
        }
    }

    with open(zattrs_path, 'w') as f:
        json.dump(zattrs, f, indent=2)

print("Metadata written to all FOV .zattrs.")

## TODO
- Apply the same NGFF reorganisation and metadata writing to the live-cell zarr stores (`live_1.zarr`, `live_2.zarr`)
- Note: run only after live-cell zarr arrays have finished compiling